# Kaggle remote run — the tuned harness on the competition checkpoint

This notebook is the first **remote** execution of the harness we have been tuning
locally. It runs the shipped `submission/` tree through `swegemma eval` on a Kaggle
notebook, with the Docker-less `subprocess` sandbox, and archives the artifacts with
the same `runs/<UTC>/report.json` + `STATUS` handoff the host trial produces.

## Before you run

1. **Accelerator** — Settings → Accelerator → `GPU L4 x4` (the scored environment).
   `GPU T4 x2` is not representative and the W4A16 checkpoint may not load on Turing.
2. **Internet** — Settings → Internet → `On`. The notebook clones this repository and
   installs the harness from the wheelhouse.
3. **Inputs** (Add Input):
   - Competition data for `gemma-4-developer-agent` (tasks, snapshots, wheels,
     graphs, embeddings, sandbox), **or** a small dataset with the same layout and only
     the snapshots you need.
   - Dataset `metric/gemma-4-developer-agent-wheelhouse` (the harness + vLLM wheels).
   - Model `google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct` version `1`.
4. **Secrets** — only for `MODEL_BACKEND = "api"`: add `OPENAI_API_KEY`
   (Add-ons → Secrets). The `local` backend needs no secret.

Run the cells top to bottom. The configuration cell is the only one you edit for a
normal run; the last cells archive, package and stop the model server.


## 1. Configuration

In [ ]:
# =============================================================================
# 1. Configuration — edit this cell, then run the notebook top to bottom.
# =============================================================================
RUN_LABEL = "kaggle_remote_01"

# Repository that carries the tuned harness (submission/, experiments/subsets/).
REPO_URL = "https://github.com/emiliodavola/kaggle-gemma-agent.git"
REPO_REF = "main"  # a branch, tag or commit sha; pin one for a reproducible run

# What to evaluate.
SUBSET = "dev-v1"  # experiments/subsets/<id>.json; None disables it
TASK_IDS: list[str] = []  # explicit ids; wins over SUBSET
MAX_TASKS = 2  # the first remote run stays small on purpose

# Per-task budgets. Competition caps: 100 tool calls / 60 min / 500 turns.
MAX_TOOL_CALLS = 100
MAX_TIME_MINUTES = 30
MAX_TURNS = 500
COMMAND_TIMEOUT_SECONDS = 300
CONCURRENCY = 1
DISPLAY = "quiet"  # quiet | single | dashboard | auto

# Model backend:
#   "local" — vLLM on this notebook's GPUs, serving the competition checkpoint.
#   "api"   — an OpenAI-compatible endpoint (Kaggle secret OPENAI_API_KEY).
MODEL_BACKEND = "local"
SERVED_MODEL_NAME = "gemma-4-31b-it-qat-w4a16-ct"
MODEL_PATH = ""  # "" = auto-detect under /kaggle/input/models
TENSOR_PARALLEL_SIZE = 0  # 0 = every visible GPU
MAX_MODEL_LEN = 32768  # the scored context ceiling
GPU_MEMORY_UTILIZATION = 0.80
VLLM_DTYPE = "auto"  # try "half" if the checkpoint does not load
VLLM_EXTRA_ARGS: list[str] = []

# "api" backend only.
API_PRESET = "opencode"  # see BACKEND_PRESETS in scripts/host-trial/run_host_trial.py
API_MODEL = "deepseek-v4.1-flash"
API_BASE_URL = ""  # "" = the preset URL

# Paths. Keep the defaults unless your inputs are mounted elsewhere.
DATA_ROOT = ""  # "" = auto-detect under /kaggle/input
WHEELHOUSE = ""  # "" = auto-detect the swegemma wheelhouse
WORK_DIR = "/kaggle/working/kaggle-remote-run"

if MODEL_BACKEND not in {"local", "api"}:
    raise ValueError("MODEL_BACKEND must be 'local' or 'api'")
if MAX_TASKS < 1:
    raise ValueError("MAX_TASKS must be at least 1")
print("run label :", RUN_LABEL)
print("backend   :", MODEL_BACKEND)
print("selection :", TASK_IDS or SUBSET or "whole corpus", "| cap", MAX_TASKS)

## 2. Environment probe

In [ ]:
# =============================================================================
# 2. Environment probe — fails loudly if the session is not what this run needs.
# =============================================================================
import importlib.metadata
import json
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

WORK = Path(WORK_DIR)
WORK.mkdir(parents=True, exist_ok=True)


def sh(argv, **kwargs):
    # Run a command, echo it, and return the CompletedProcess.
    print("$ " + " ".join(str(a) for a in argv), flush=True)
    kwargs.setdefault("text", True)
    return subprocess.run([str(a) for a in argv], **kwargs)


def bounded_glob(root, pattern, max_depth=3):
    # Yield files matching pattern up to max_depth directory levels below root.
    root = Path(root)
    if not root.is_dir():
        return
    stack = [(root, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            for entry in current.iterdir():
                if entry.is_file() and entry.match(pattern):
                    yield entry
                elif entry.is_dir() and depth < max_depth and not entry.is_symlink():
                    stack.append((entry, depth + 1))
        except (PermissionError, OSError):
            continue


print("python :", sys.version.split()[0], "|", sys.executable)
print("cwd    :", os.getcwd())

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
print("gpus   :\n" + (smi.stdout.strip() or smi.stderr.strip() or "none"))

usage = shutil.disk_usage(WORK)
print(f"work   : {usage.free / 2**30:.1f} GiB free of {usage.total / 2**30:.1f} GiB at {WORK}")

## 3. Install the harness

In [ ]:
# =============================================================================
# 3. Harness install — the wheelhouse pins the scoring stack; PyPI fills the gaps.
# =============================================================================
def unpack_wheelhouse(archive):
    target = WORK / "wheelhouse"
    if list(target.glob("swegemma-*.whl")):
        return target
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(target)
    print(f"unpacked {archive} -> {target}")
    return target


def find_wheelhouse():
    if WHEELHOUSE:
        candidate = Path(WHEELHOUSE)
        if candidate.is_dir() and list(candidate.glob("swegemma-*.whl")):
            return candidate
        if candidate.is_file() and candidate.suffix == ".zip":
            return unpack_wheelhouse(candidate)
        raise FileNotFoundError(f"WHEELHOUSE does not hold swegemma-*.whl: {candidate}")
    for wheel in bounded_glob("/kaggle/input", "swegemma-*.whl", max_depth=4):
        return wheel.parent
    for archive in bounded_glob("/kaggle/input", "*wheelhouse*.zip", max_depth=3):
        return unpack_wheelhouse(archive)
    raise FileNotFoundError(
        "No swegemma wheelhouse under /kaggle/input. Attach the dataset "
        "'metric/gemma-4-developer-agent-wheelhouse' or set WHEELHOUSE."
    )


WHEELHOUSE_DIR = find_wheelhouse()
print("wheelhouse:", WHEELHOUSE_DIR)

sh(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-input",
        "--quiet",
        "--find-links",
        WHEELHOUSE_DIR,
        "swegemma==0.2.7",
        "transformers==5.13.1",
        "litellm>=1.83.7,<=1.83.14",
    ],
    check=True,
)

if MODEL_BACKEND == "local":
    try:
        installed_vllm = importlib.metadata.version("vllm")
    except importlib.metadata.PackageNotFoundError:
        installed_vllm = None
    if installed_vllm is None:
        # vLLM pins its own torch; a session-local torch change is expected and fine.
        sh(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-input",
                "--quiet",
                "--find-links",
                WHEELHOUSE_DIR,
                "vllm==0.19.1",
            ],
            check=True,
        )
    else:
        print("vllm already installed:", installed_vllm)

for dist in (
    "swegemma",
    "adk-submission",
    "adk-eval-core",
    "google-adk",
    "litellm",
    "transformers",
    "torch",
    "vllm",
):
    try:
        print(f"{dist:<15} {importlib.metadata.version(dist)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{dist:<15} MISSING")

## 4. Clone the repository and run the submission gate

In [ ]:
# =============================================================================
# 4. Repository + submission gate — the same gate CI runs, before any GPU time.
# =============================================================================
REPO_DIR = WORK / "repo"
if not (REPO_DIR / ".git").is_dir():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    sh(["git", "-C", REPO_DIR, "init", "--quiet"], check=True)
    sh(["git", "-C", REPO_DIR, "remote", "add", "origin", REPO_URL], check=True)

sh(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_REF], check=True)
sh(["git", "-C", REPO_DIR, "checkout", "--force", "FETCH_HEAD"], check=True)

COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print("repo commit:", COMMIT)

GATE_ENV = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src")}
gate = sh(
    [sys.executable, "-m", "kaggle_gemma_agent.pack", "submission", "--check"],
    cwd=REPO_DIR,
    env=GATE_ENV,
)
if gate.returncode != 0:
    raise SystemExit("submission gate failed: fix submission/ before spending GPU time")

SUBMISSION_DIR = REPO_DIR / "submission"
DECLARED_MODEL = re.search(
    r"^model:\s*(\S+)",
    (SUBMISSION_DIR / "agent.yaml").read_text(encoding="utf-8"),
    re.MULTILINE,
).group(1)
print("declared model:", DECLARED_MODEL)

## 5. Competition data and task selection

In [ ]:
# =============================================================================
# 5. Competition data + task selection — validates only the tasks this run needs.
# =============================================================================
def find_data_root():
    if DATA_ROOT:
        return Path(DATA_ROOT)
    candidates = []
    for tasks in bounded_glob("/kaggle/input", "tasks.jsonl", max_depth=3):
        root = tasks.parent
        if (root / "snapshots").is_dir() or (root / "wheels").is_dir():
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError(
            "No competition data root under /kaggle/input (need tasks.jsonl next to "
            "snapshots/ and wheels/). Attach the competition data or set DATA_ROOT."
        )
    return sorted(candidates, key=lambda path: len(str(path)))[0]


DATA = find_data_root()
TASKS_FILE = DATA / "tasks.jsonl"
SNAPSHOTS_DIR = DATA / "snapshots"
WHEELS_COMPETITION = DATA / "wheels"
GRAPHS_DIR = DATA / "graphs"
EMBEDDINGS_DIR = DATA / "embeddings"
print("data root:", DATA)

missing = [path for path in (TASKS_FILE, SNAPSHOTS_DIR, WHEELS_COMPETITION) if not path.exists()]
if missing:
    raise FileNotFoundError(
        "competition data is incomplete; missing: " + ", ".join(str(p) for p in missing)
    )
if not GRAPHS_DIR.is_dir() or not EMBEDDINGS_DIR.is_dir():
    print("warning: graphs/ or embeddings/ missing; the code-intelligence tools degrade")

corpus = {}
with TASKS_FILE.open(encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            record = json.loads(line)
            corpus[record["instance_id"]] = record

if TASK_IDS:
    selected = list(TASK_IDS)
elif SUBSET:
    subset_path = REPO_DIR / "experiments" / "subsets" / f"{SUBSET}.json"
    selected = json.loads(subset_path.read_text(encoding="utf-8"))["task_ids"]
else:
    selected = list(corpus)
selected = selected[:MAX_TASKS]
if not selected:
    raise ValueError("no task selected; set TASK_IDS, SUBSET or MAX_TASKS")

unknown = [task_id for task_id in selected if task_id not in corpus]
if unknown:
    raise ValueError(f"task ids not in tasks.jsonl: {unknown}")

absent = []
for task_id in selected:
    if not any((SNAPSHOTS_DIR / f"{task_id}{suffix}").is_file() for suffix in (".tgz", ".tar.gz")):
        absent.append(task_id)
if absent:
    raise FileNotFoundError(
        "snapshots missing for "
        + ", ".join(absent)
        + ". Attach a dataset with those snapshots or fetch them with "
        + "'kaggle competitions download -c gemma-4-developer-agent -f snapshots/<id>.tgz'."
    )

print(f"tasks ({len(selected)}):")
for task_id in selected:
    record = corpus[task_id]
    print(f"  {task_id:<18} {record['repo']:<18} {record['base_commit'][:10]}")

## 6. Start the model

In [ ]:
# =============================================================================
# 6. Model backend — vLLM with the scoring wiring, or an OpenAI-compatible API.
# =============================================================================
MODEL_SERVER = None
HEADER_PROXY = None
API_KEY = "EMPTY"

sys.path.insert(0, str(REPO_DIR / "scripts" / "host-trial"))
from run_host_trial import (  # noqa: E402  (import after the clone exists)
    BACKEND_PRESETS,
    DEFAULT_TRIAL_SESSION,
    HeaderInjectingProxy,
    smoke_test_backend,
)


def secret(name):
    value = os.environ.get(name, "")
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""


if MODEL_BACKEND == "local":
    import torch
    from adk_submission.server import VllmConfig, spawn_vllm_server
    from swegemma.models.discovery import resolve_local_model_path

    resolved = Path(MODEL_PATH) if MODEL_PATH else Path(resolve_local_model_path(DECLARED_MODEL))
    if not (resolved / "config.json").is_file():
        found = [
            path.parent
            for path in bounded_glob("/kaggle/input", "config.json", max_depth=6)
            if DECLARED_MODEL.lower() in str(path.parent).lower()
        ]
        if not found:
            raise FileNotFoundError(
                f"model {DECLARED_MODEL} not found under /kaggle/input. Attach the Kaggle "
                "Model 'google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct' (version 1) "
                "or set MODEL_PATH."
            )
        resolved = found[0]

    gpu_count = torch.cuda.device_count()
    if gpu_count < 1:
        raise RuntimeError("MODEL_BACKEND='local' needs a GPU accelerator; none is visible")
    parallel = TENSOR_PARALLEL_SIZE or gpu_count
    print("model:", resolved)
    print(f"gpus: {gpu_count} | tensor_parallel_size: {parallel}")

    config = VllmConfig(
        model=str(resolved),
        tensor_parallel_size=parallel,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        max_model_len=MAX_MODEL_LEN,
        dtype=VLLM_DTYPE,
        tool_call_parser="gemma4",
        reasoning_parser="gemma4",
        default_chat_template_kwargs={"enable_thinking": True},
        enable_lora=True,
        max_loras=8,
        max_lora_rank=128,
        extra_args=["--served-model-name", SERVED_MODEL_NAME, *VLLM_EXTRA_ARGS],
        startup_timeout=1800,
    )
    MODEL_SERVER = spawn_vllm_server(config)
    MODEL_SERVER.start()
    BASE_URL = MODEL_SERVER.base_url
    SERVED_MODEL = SERVED_MODEL_NAME
    BACKEND_LABEL = SERVED_MODEL_NAME
    BACKEND_NAME = "vllm-kaggle"
else:
    API_KEY = secret("OPENAI_API_KEY")
    if not API_KEY:
        raise RuntimeError(
            "MODEL_BACKEND='api' needs an OPENAI_API_KEY Kaggle secret "
            "(Add-ons -> Secrets) or the OPENAI_API_KEY environment variable."
        )
    upstream = API_BASE_URL or BACKEND_PRESETS[API_PRESET]
    HEADER_PROXY = HeaderInjectingProxy(
        upstream, secret("HARNESS_TRIAL_SESSION") or DEFAULT_TRIAL_SESSION
    )
    BASE_URL = HEADER_PROXY.start()
    SERVED_MODEL = API_MODEL
    BACKEND_LABEL = API_MODEL
    BACKEND_NAME = API_PRESET
    print("upstream:", upstream)

print("base url:", BASE_URL)
print("served model:", SERVED_MODEL)
print("smoke:", smoke_test_backend(BASE_URL, SERVED_MODEL, API_KEY))

## 7. Evaluate

In [ ]:
# =============================================================================
# 7. Evaluation — swegemma eval with the Docker-less subprocess sandbox.
# =============================================================================
MODELS_YAML = WORK / "models.yaml"
MODELS_YAML.write_text(
    "models:\n"
    f"  {DECLARED_MODEL}:\n"
    f"    path: openai/{SERVED_MODEL}\n"
    f"    api_base: {BASE_URL}\n"
    "    api_key: EMPTY\n",
    encoding="utf-8",
)
print(MODELS_YAML.read_text(encoding="utf-8"))

RESULTS_DIR = WORK / "results" / RUN_LABEL
if RESULTS_DIR.exists():
    raise SystemExit(
        f"{RESULTS_DIR} already exists; change RUN_LABEL so two runs never share a results dir"
    )

EVAL_ENV = {
    **os.environ,
    "PYTHONPATH": str(REPO_DIR / "src"),
    "KAGGLE_SANDBOX_DIR": str(DATA),
}
EVAL_ARGV = [
    sys.executable,
    "-m",
    "swegemma",
    "eval",
    "--tasks",
    TASKS_FILE,
    "--snapshots-dir",
    SNAPSHOTS_DIR,
    "--submission-dir",
    SUBMISSION_DIR,
    "--task-ids",
    *selected,
    "--results-dir",
    RESULTS_DIR,
    "--sandbox",
    "subprocess",
    "--models-yaml",
    MODELS_YAML,
    "--max-tool-calls",
    MAX_TOOL_CALLS,
    "--max-time-minutes",
    MAX_TIME_MINUTES,
    "--max-turns",
    MAX_TURNS,
    "--timeout-seconds",
    COMMAND_TIMEOUT_SECONDS,
    "--concurrency",
    CONCURRENCY,
    "--display",
    DISPLAY,
]

started = time.time()
completed = sh(EVAL_ARGV, cwd=WORK, env=EVAL_ENV)
print(f"eval exit code: {completed.returncode} in {time.time() - started:.0f}s")
if completed.returncode != 0:
    print("eval failed; read the output above. The model server stays up for a retry.")

## 8. Archive and report

In [ ]:
# =============================================================================
# 8. Archive + report — the same runs/<UTC>/ handoff the host trial produces.
# =============================================================================
if not RESULTS_DIR.is_dir():
    raise SystemExit("no results directory; run the evaluation cell first")

RUNS_ROOT = WORK / "artifacts" / "runs"
ARCHIVE_ENV = {
    **os.environ,
    "PYTHONPATH": str(REPO_DIR / "src"),
    "ROBOTINA_HERMES_MODEL": BACKEND_LABEL,
    "ROBOTINA_HERMES_MODEL_BASE_URL": BASE_URL,
}
ARCHIVE_ARGV = [
    sys.executable,
    "-m",
    "kaggle_gemma_agent.harness_runs",
    "archive",
    RESULTS_DIR,
    "--runs-root",
    RUNS_ROOT,
    "--tasks",
    TASKS_FILE,
    "--backend",
    BACKEND_LABEL,
    "--backend-name",
    BACKEND_NAME,
    "--model",
    SERVED_MODEL,
    "--sandbox-deps-mode",
    "faithful",
    "--no-journal",
]
if SUBSET:
    ARCHIVE_ARGV += ["--subset", SUBSET]

archived = sh(ARCHIVE_ARGV, cwd=REPO_DIR, env=ARCHIVE_ENV)
if archived.returncode != 0:
    raise SystemExit("archive failed; the raw results/ directory is still complete")

RUN_DIR = max(RUNS_ROOT.iterdir(), key=lambda path: path.stat().st_mtime)
print()
print((RUN_DIR / "STATUS").read_text(encoding="utf-8"))

records = [
    json.loads(line)
    for line in (RESULTS_DIR / "task_results.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"{'instance_id':<20}{'resolved':<10}{'tools':<8}error")
for record in records:
    print(
        f"{record.get('instance_id', '?'):<20}"
        f"{record.get('resolved')!s:<10}"
        f"{record.get('tool_calls')!s:<8}"
        f"{record.get('error') or ''}"
    )

## 9. Package the artifacts

In [ ]:
# =============================================================================
# 9. Package — one zip with results/, runs/ and the run metadata.
# =============================================================================
BUNDLE = Path("/kaggle/working") / f"{RUN_LABEL}.zip"
META = {
    "run_label": RUN_LABEL,
    "repo": REPO_URL,
    "repo_ref": REPO_REF,
    "repo_commit": COMMIT,
    "model_backend": MODEL_BACKEND,
    "served_model": SERVED_MODEL,
    "backend_name": BACKEND_NAME,
    "base_url": BASE_URL,
    "data_root": str(DATA),
    "task_ids": selected,
    "budgets": {
        "max_tool_calls": MAX_TOOL_CALLS,
        "max_time_minutes": MAX_TIME_MINUTES,
        "max_turns": MAX_TURNS,
        "command_timeout_seconds": COMMAND_TIMEOUT_SECONDS,
    },
    "eval_exit_code": completed.returncode,
    "archived_run": str(RUN_DIR),
}
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as bundle:
    for root in (RESULTS_DIR, RUNS_ROOT):
        for path in root.rglob("*"):
            if path.is_file():
                bundle.write(path, path.relative_to(WORK))
    bundle.writestr("run_meta.json", json.dumps(META, indent=2))

print(f"{BUNDLE} ({BUNDLE.stat().st_size / 2**20:.1f} MiB)")
print("Download it from the notebook output panel.")

## 10. Stop the model server

In [ ]:
# =============================================================================
# 10. Stop the model server — always run this when you are done with the session.
# =============================================================================
if MODEL_SERVER is not None:
    MODEL_SERVER.stop()
    MODEL_SERVER = None
    print("vLLM stopped")
if HEADER_PROXY is not None:
    HEADER_PROXY.stop()
    HEADER_PROXY = None
    print("header proxy stopped")

## What to bring back

- `<RUN_LABEL>.zip` from the notebook output panel. It holds `results/<RUN_LABEL>/`
  (summary, task results, patches, test outputs, traces, logs), `artifacts/runs/<UTC>/`
  (`report.json`, `STATUS`, per-task artifacts) and `run_meta.json`.
- Copy the archive into the repository's `runs/` workflow on the host and analyse it
  with `docs/run-reports.md`; do not paste logs into chat.

## Reading the result

- `resolved` is the SWE-bench verdict for one task; the run rate is resolved / tasks.
- `STATUS` is the six-line handoff: backend, tasks/resolved/rate, budget use, failure
  kinds, delivery scope and artifact paths.
- A `test_failure` means the patch was applied and pytest failed; an `error` means the
  harness could not finish the task (snapshot, sandbox, model or budget). Read the
  per-task `error` before attributing anything to the harness.

## Known limits of this notebook

- The `subprocess` sandbox is not the 4 GiB / 2 vCPU Docker container; it isolates
  files and the process group, not memory or CPU.
- JUnit XML stays inside the sandbox and is lost on teardown, exactly as in a host
  trial, so the verdict comes from `task_results.jsonl`.
- The notebook image's Python and wheels may differ from the scored image; a wheel
  that is missing here is not necessarily missing at scoring time.
- The first remote run is evidence, not an estimate: a two-task run on `dev-v1` says
  nothing about the hidden test set.
